# Prepare 2023–2024 Heating Season Intake Features & Slow-Resolution Labels

**Cohort Population**: NYC 311 `HEAT/HOT WATER` service requests created between October 1, 2023 and May 31, 2024 (235,654 records).

**Frozen Target Threshold**: $\tau = 62.34933333333333$ hours ($\approx 2.60$ days).
- Derived as the empirical 90th percentile of quiet-window closed tickets in January 2024.
- Frozen here as a project-defined experimental benchmark (not a published City SLA).

**Column Separation**:
- `AUDIT_COLUMNS`: `["unique_key", "created_date", "split"]` (tracking and partitioning only; never fed to model).
- `MODEL_INPUT_COLUMNS`: `["complaint_type", "descriptor", "descriptor_2", "location_type", "borough", "open_data_channel_type", "agency", "created_hour", "created_day_of_week", "created_is_weekend"]`.
- `TARGET_COLUMNS`: Stored in a quarantined label artifact (`data/labelled/`).

**Rules this notebook obeys**:
1. **Zero Outcome Leakage**: Outcome fields (`closed_date`, `status`) never enter the feature matrix.
2. **Deterministic Labeling for Mature Open Cases**: The 322 open/in-progress tickets have follow-up $\Delta t > 15,000\text{ hours} \gg \tau$ and are labeled Slow ($Y = 1$).
3. **Exact Chronological Partitions**: Non-overlapping splits: Train (Oct–Dec 2023), Validation (Jan–Feb 2024), Test (Mar–May 2024).
4. **Test Set Blindness**: Label distributions are audited strictly for Train and Validation. Test labels remain unsummarized until final model evaluation.
5. **Privacy Strictness**: Street addresses, coordinate fields, and `incident_zip` are excluded by design.

In [1]:
import hashlib
import json
from pathlib import Path
import time

import pandas as pd

from resolveiq.config import REPO_ROOT

RAW_DATA_PATH = REPO_ROOT / "data" / "raw" / "heat_2023_10_2024_05_season.json"
RAW_META_PATH = REPO_ROOT / "data" / "raw" / "heat_2023_10_2024_05_season.meta.json"

assert RAW_DATA_PATH.exists(), f"Raw data missing at {RAW_DATA_PATH}"
assert RAW_META_PATH.exists(), f"Metadata sidecar missing at {RAW_META_PATH}"

# 1. Verify file against its cryptographic SHA-256 sidecar
sidecar = json.loads(RAW_META_PATH.read_text(encoding="utf-8"))
raw_bytes = RAW_DATA_PATH.read_bytes()
computed_sha256 = hashlib.sha256(raw_bytes).hexdigest()

assert computed_sha256 == sidecar["sha256"], (
    f"SHA-256 mismatch!\nExpected: {sidecar['sha256']}\nComputed: {computed_sha256}"
)
print(f"[PASS] Cryptographic integrity verified: SHA-256 {computed_sha256}")

# 2. Load into DataFrame
t0 = time.perf_counter()
raw_records = json.loads(raw_bytes.decode("utf-8"))
df_raw = pd.DataFrame(raw_records)
elapsed = time.perf_counter() - t0

assert len(df_raw) == 235654, f"Expected 235,654 rows, found {len(df_raw):,}"
assert len(df_raw["unique_key"].unique()) == 235654, "Duplicate unique_key detected!"

print(f"[PASS] Loaded {len(df_raw):,} records into DataFrame in {elapsed:.2f}s")
print(f"Memory footprint : {df_raw.memory_usage(deep=True).sum() / (1024 * 1024):.1f} MB")
print(f"Columns present  : {list(df_raw.columns)}")

[PASS] Cryptographic integrity verified: SHA-256 773af945f3b2defac1291597494960da706c9fe13ac44bc9394ffa84c92c8ff2
[PASS] Loaded 235,654 records into DataFrame in 0.73s
Memory footprint : 151.8 MB
Columns present  : ['unique_key', 'created_date', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'borough', 'open_data_channel_type', 'agency', 'closed_date', 'status']


## Section 2: Non-Overlapping Chronological Partitions

We partition the 235,654 records into three contiguous, non-overlapping half-open intervals:
- **`train`**: `[2023-10-01T00:00:00, 2024-01-01T00:00:00)` (Oct–Dec 2023; 3 months)
- **`val`**: `[2024-01-01T00:00:00, 2024-03-01T00:00:00)` (Jan–Feb 2024; 2 months, peak winter)
- **`test`**: `[2024-03-01T00:00:00, 2024-06-01T00:00:00)` (Mar–May 2024; 3 months, spring taper)

In [2]:
# 1. Define non-overlapping masks using ISO-8601 half-open intervals
train_mask = (df_raw["created_date"] >= "2023-10-01T00:00:00") & (df_raw["created_date"] < "2024-01-01T00:00:00")
val_mask = (df_raw["created_date"] >= "2024-01-01T00:00:00") & (df_raw["created_date"] < "2024-03-01T00:00:00")
test_mask = (df_raw["created_date"] >= "2024-03-01T00:00:00") & (df_raw["created_date"] < "2024-06-01T00:00:00")

# 2. Assign split tag
df_raw["split"] = None
df_raw.loc[train_mask, "split"] = "train"
df_raw.loc[val_mask, "split"] = "val"
df_raw.loc[test_mask, "split"] = "test"

# 3. Assert exhaustiveness and mutual exclusivity
assert df_raw["split"].isnull().sum() == 0, "Unassigned records detected!"
assert (train_mask & val_mask).sum() == 0, "Train and Val overlap!"
assert (val_mask & test_mask).sum() == 0, "Val and Test overlap!"
assert (train_mask & test_mask).sum() == 0, "Train and Test overlap!"
assert len(df_raw) == 235654, f"Row count changed: {len(df_raw)}"

# 4. Report partition breakdown
split_summary = []
for split_name in ["train", "val", "test"]:
    subset = df_raw[df_raw["split"] == split_name]
    split_summary.append({
        "split": split_name,
        "rows": len(subset),
        "share": f"{len(subset) / len(df_raw):.1%}",
        "min_created": subset["created_date"].min(),
        "max_created": subset["created_date"].max(),
    })

df_split_summary = pd.DataFrame(split_summary)
print("[PASS] Chronological splits assigned with zero overlap and zero unassigned rows:\n")
print(df_split_summary.to_string(index=False))

[PASS] Chronological splits assigned with zero overlap and zero unassigned rows:

split   rows share             min_created             max_created
train 106188 45.1% 2023-10-01T00:07:32.000 2023-12-31T23:57:56.000
  val  81641 34.6% 2024-01-01T00:01:45.000 2024-02-29T23:59:27.000
 test  47825 20.3% 2024-03-01T00:02:45.000 2024-05-31T23:19:34.000


## Section 3: Audit of Edge Cases (322 Unclosed & 1 Unspecified Record)

We verify:
1. That all 322 unclosed records (`closed_date` is null) have elapsed follow-up time $\Delta t \gg 62.34933333333333$ hours, confirming they are deterministically Slow ($Y = 1$).
2. The exact fields and resolution duration of the single `Unspecified` record with a populated `closed_date`.

In [3]:
# 1. Audit the 322 unclosed records (Open and In Progress)
df_unclosed = df_raw[df_raw["closed_date"].isnull()].copy()
assert len(df_unclosed) == 322, f"Expected 322 unclosed rows, found {len(df_unclosed)}"

# Calculate elapsed follow-up hours from intake to now
now_utc = pd.Timestamp.now(tz="UTC")
created_utc = pd.to_datetime(df_unclosed["created_date"], utc=True)
df_unclosed["follow_up_hours"] = (now_utc - created_utc).dt.total_seconds() / 3600.0

min_follow_up = df_unclosed["follow_up_hours"].min()
max_follow_up = df_unclosed["follow_up_hours"].max()

print("=== 1. Unclosed Tickets Follow-Up Audit ===")
print(f"Total unclosed rows     : {len(df_unclosed)}")
print(f"Status breakdown        :\n{df_unclosed['status'].value_counts().to_dict()}")
print(f"Split breakdown         :\n{df_unclosed['split'].value_counts().to_dict()}")
print(f"Minimum follow-up time  : {min_follow_up:,.1f} hours ({min_follow_up / 24:.0f} days)")
print(f"Maximum follow-up time  : {max_follow_up:,.1f} hours ({max_follow_up / 24:.0f} days)")

FROZEN_TAU_HOURS = 62.34933333333333
assert min_follow_up > FROZEN_TAU_HOURS, (
    f"Right-censored ticket found! Min follow-up {min_follow_up} < tau {FROZEN_TAU_HOURS}"
)
print(f"[PASS] All 322 unclosed tickets exceed tau ({FROZEN_TAU_HOURS:.2f}h). Deterministically Slow (Y=1).\n")

# 2. Audit the single Unspecified record
df_unspec = df_raw[df_raw["status"] == "Unspecified"].copy()
assert len(df_unspec) == 1, f"Expected 1 Unspecified row, found {len(df_unspec)}"
unspec_row = df_unspec.iloc[0]

unspec_created = pd.to_datetime(unspec_row["created_date"])
unspec_closed = pd.to_datetime(unspec_row["closed_date"])
unspec_duration_hours = (unspec_closed - unspec_created).total_seconds() / 3600.0

print("=== 2. Single 'Unspecified' Record Audit ===")
print(f"Unique Key       : {unspec_row['unique_key']}")
print(f"Created Date     : {unspec_row['created_date']}")
print(f"Closed Date      : {unspec_row['closed_date']}")
print(f"Assigned Split   : {unspec_row['split']}")
print(f"Descriptor       : {unspec_row['descriptor']}")
print(f"Resolution Hours : {unspec_duration_hours:.2f} hours")

assert unspec_duration_hours >= 0, f"Negative duration detected: {unspec_duration_hours}"
print(f"[PASS] Non-negative duration verified ({unspec_duration_hours:.2f}h). Eligible for modeling.")

=== 1. Unclosed Tickets Follow-Up Audit ===
Total unclosed rows     : 322
Status breakdown        :
{'Open': 211, 'In Progress': 111}
Split breakdown         :
{'train': 198, 'val': 104, 'test': 20}
Minimum follow-up time  : 20,589.7 hours (858 days)
Maximum follow-up time  : 26,177.2 hours (1091 days)
[PASS] All 322 unclosed tickets exceed tau (62.35h). Deterministically Slow (Y=1).

=== 2. Single 'Unspecified' Record Audit ===
Unique Key       : 59436684
Created Date     : 2023-10-02T14:55:22.000
Closed Date      : 2023-10-03T00:00:00.000
Assigned Split   : train
Descriptor       : Apartment Only
Resolution Hours : 9.08 hours
[PASS] Non-negative duration verified (9.08h). Eligible for modeling.


## Section 4: Target Labeling ($Y \in \{0, 1\}$)

We compute the slow-resolution label $Y$ using the frozen threshold:
$$\tau = 62.34933333333333\text{ hours}$$

- **Closed & Unspecified tickets** ($N = 235,332$): Labeled by observed duration $T_{\text{resolve}} > \tau$.
- **Open & In Progress tickets** ($N = 322$): Labeled deterministically as $Y = 1$ because elapsed follow-up $\Delta t > 20,000\text{h} \gg \tau$.

In [5]:
FROZEN_TAU_HOURS = 62.34933333333333

# 1. Initialize label dataframe with unique_key anchor
df_labels = pd.DataFrame()
df_labels["unique_key"] = df_raw["unique_key"]
df_labels["status"] = df_raw["status"]
df_labels["non_standard_status"] = df_raw["status"] == "Unspecified"
df_labels["resolution_hours"] = None
df_labels["slow_resolution_label"] = None
df_labels["is_clock_anomaly"] = False
df_labels["eligible_for_modeling"] = False

# 2. Process records with closed_date present (Closed + Unspecified)
closed_mask = df_raw["closed_date"].notnull()
created_dt = pd.to_datetime(df_raw.loc[closed_mask, "created_date"])
closed_dt = pd.to_datetime(df_raw.loc[closed_mask, "closed_date"])

duration_hours = (closed_dt - created_dt).dt.total_seconds() / 3600.0

# Isolate clock anomalies (closed_date < created_date)
anomaly_mask_closed = duration_hours < 0
num_anomalies = anomaly_mask_closed.sum()

# Map anomaly mask back to df_labels indices
anomaly_indices = df_raw.loc[closed_mask].index[anomaly_mask_closed]
valid_closed_indices = df_raw.loc[closed_mask].index[~anomaly_mask_closed]

df_labels.loc[anomaly_indices, "is_clock_anomaly"] = True
df_labels.loc[anomaly_indices, "eligible_for_modeling"] = False
# resolution_hours and slow_resolution_label remain None for anomalies

# Assign valid closed records
valid_durations = duration_hours[~anomaly_mask_closed]
df_labels.loc[valid_closed_indices, "resolution_hours"] = valid_durations.round(4)
df_labels.loc[valid_closed_indices, "slow_resolution_label"] = (valid_durations > FROZEN_TAU_HOURS).astype(int)
df_labels.loc[valid_closed_indices, "eligible_for_modeling"] = True

# 3. Assign mature unclosed records (Open + In Progress)
unclosed_mask = df_raw["closed_date"].isnull()
unclosed_indices = df_raw.index[unclosed_mask]

df_labels.loc[unclosed_indices, "slow_resolution_label"] = 1
df_labels.loc[unclosed_indices, "eligible_for_modeling"] = True
# resolution_hours remains None for unclosed tickets

# 4. Inspect the 162 clock anomalies
df_anomalies = df_raw.loc[anomaly_indices]
negative_hours = duration_hours[anomaly_mask_closed]

print("=== Clock Anomaly Quarantine Audit ===")
print(f"Total closed records evaluated       : {closed_mask.sum():,}")
print(f"Valid closed records                 : {len(valid_closed_indices):,} (assigned duration & label)")
print(f"Quarantined clock anomalies          : {num_anomalies:,} (closed_date < created_date)")
print(f"  Anomaly min duration               : {negative_hours.min():.2f} hours")
print(f"  Anomaly median duration            : {negative_hours.median():.2f} hours")
print(f"  Anomaly split distribution         : {dict(df_anomalies['split'].value_counts())}")

# 5. Integrity assertions
assert len(df_labels) == 235654, f"Row count changed: {len(df_labels)}"
assert df_labels["is_clock_anomaly"].sum() == num_anomalies
assert df_labels["eligible_for_modeling"].sum() == 235654 - num_anomalies

# Check valid records
df_eligible = df_labels[df_labels["eligible_for_modeling"]]
assert df_eligible["slow_resolution_label"].isnull().sum() == 0, "Missing label among eligible records!"
assert set(df_eligible["slow_resolution_label"].unique()) == {0, 1}, "Eligible labels not binary {0, 1}!"

# Verify the single Unspecified record
unspec_row = df_labels[df_labels["non_standard_status"]].iloc[0]
assert unspec_row["eligible_for_modeling"] == True
assert unspec_row["slow_resolution_label"] == 0

print("\n=== Target Dataset Eligibility Summary ===")
print(f"Total cohort records                 : {len(df_labels):,}")
print(f"Eligible for modeling                : {df_labels['eligible_for_modeling'].sum():,} ({df_labels['eligible_for_modeling'].mean():.2%})")
print(f"  - Valid closed tickets             : {len(valid_closed_indices):,}")
print(f"  - Deterministic open tickets (Y=1) : {unclosed_mask.sum():,}")
print(f"Quarantined / Excluded from modeling : {num_anomalies:,} ({num_anomalies / len(df_labels):.2%})")
print(f"\n[PASS] Target generation and anomaly quarantine complete.")

=== Clock Anomaly Quarantine Audit ===
Total closed records evaluated       : 235,332
Valid closed records                 : 235,170 (assigned duration & label)
Quarantined clock anomalies          : 162 (closed_date < created_date)
  Anomaly min duration               : -20.31 hours
  Anomaly median duration            : -8.50 hours
  Anomaly split distribution         : {'train': np.int64(162)}

=== Target Dataset Eligibility Summary ===
Total cohort records                 : 235,654
Eligible for modeling                : 235,492 (99.93%)
  - Valid closed tickets             : 235,170
  - Deterministic open tickets (Y=1) : 322
Quarantined / Excluded from modeling : 162 (0.07%)

[PASS] Target generation and anomaly quarantine complete.


## Section 5: Intake Feature Matrix & Zero-Leakage Quarantine

We extract strictly pre-intake features and derived calendar features, enforcing complete quarantine against outcome leakage and private geographic identifiers.

- **`AUDIT_COLUMNS`**: `["unique_key", "created_date", "split"]` (tracking and partition routing only; never fed to model).
- **`MODEL_INPUT_COLUMNS`**: `["complaint_type", "descriptor", "descriptor_2", "location_type", "borough", "open_data_channel_type", "agency", "created_hour", "created_day_of_week", "created_is_weekend"]`.

In [6]:
# 1. Define explicit column groups
AUDIT_COLUMNS = ["unique_key", "created_date", "split"]

INTAKE_RAW_COLUMNS = [
    "complaint_type",
    "descriptor",
    "descriptor_2",
    "location_type",
    "borough",
    "open_data_channel_type",
    "agency",
]

# 2. Derive temporal intake features (strictly from intake timestamp)
created_dt = pd.to_datetime(df_raw["created_date"])
created_hour = created_dt.dt.hour
created_day_of_week = created_dt.dt.dayofweek
created_is_weekend = (created_day_of_week >= 5).astype(int)

# 3. Assemble df_features
df_features = pd.DataFrame()

# Audit columns
for col in AUDIT_COLUMNS:
    df_features[col] = df_raw[col]

# Raw intake features
for col in INTAKE_RAW_COLUMNS:
    df_features[col] = df_raw[col]

# Derived calendar features
df_features["created_hour"] = created_hour
df_features["created_day_of_week"] = created_day_of_week
df_features["created_is_weekend"] = created_is_weekend

MODEL_INPUT_COLUMNS = INTAKE_RAW_COLUMNS + ["created_hour", "created_day_of_week", "created_is_weekend"]

# 4. Zero-Leakage and Privacy Assertions
FORBIDDEN_OUTCOME_FIELDS = [
    "closed_date",
    "status",
    "resolution_description",
    "resolution_hours",
    "slow_resolution_label",
    "due_date",
    "resolution_action_updated_date",
]

FORBIDDEN_PII_GEOGRAPHY = [
    "incident_address",
    "incident_zip",
    "street_name",
    "latitude",
    "longitude",
    "x_coordinate_state_plane",
    "y_coordinate_state_plane",
]

for forbidden in FORBIDDEN_OUTCOME_FIELDS:
    assert forbidden not in df_features.columns, f"LEAKAGE DETECTED: {forbidden} present in df_features!"

for forbidden in FORBIDDEN_PII_GEOGRAPHY:
    assert forbidden not in df_features.columns, f"PRIVACY VIOLATION: {forbidden} present in df_features!"

assert len(df_features) == 235654, f"Row count changed: {len(df_features)}"
assert list(df_features.columns) == AUDIT_COLUMNS + MODEL_INPUT_COLUMNS

print("[PASS] Intake feature matrix constructed with zero leakage.")
print(f"Total rows           : {len(df_features):,}")
print(f"Audit columns        : {AUDIT_COLUMNS}")
print(f"Model input columns  : {MODEL_INPUT_COLUMNS}")
print(f"Feature matrix memory: {df_features.memory_usage(deep=True).sum() / (1024 * 1024):.1f} MB")

[PASS] Intake feature matrix constructed with zero leakage.
Total rows           : 235,654
Audit columns        : ['unique_key', 'created_date', 'split']
Model input columns  : ['complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'borough', 'open_data_channel_type', 'agency', 'created_hour', 'created_day_of_week', 'created_is_weekend']
Feature matrix memory: 138.8 MB


## Section 6: Split Health & Class Balance (Train & Validation Only)

We audit class balance ($P(Y=1)$) and resolution distributions exclusively for `train` and `val`. 

**Test Set Blindness Rule**: In accordance with evaluation standards, test labels remain sealed and uninspected until final model scoring.

In [7]:
# 1. Join split from features to labels for evaluation
df_combined = df_labels.copy()
df_combined["split"] = df_features["split"]
df_combined["borough"] = df_features["borough"]

# 2. Audit Train Partition
train_eligible = df_combined[(df_combined["split"] == "train") & (df_combined["eligible_for_modeling"])]
train_total = len(train_eligible)
train_slow = (train_eligible["slow_resolution_label"] == 1).sum()
train_not_slow = (train_eligible["slow_resolution_label"] == 0).sum()
train_slow_rate = train_slow / train_total
train_median_hours = train_eligible["resolution_hours"].dropna().median()

print("=== Train Partition Health (Oct–Dec 2023) ===")
print(f"Eligible records       : {train_total:,} (162 clock anomalies excluded)")
print(f"Not Slow (Y = 0)       : {train_not_slow:,} ({1 - train_slow_rate:.1%})")
print(f"Slow     (Y = 1)       : {train_slow:,} ({train_slow_rate:.1%})")
print(f"Median resolution time : {train_median_hours:.1f} hours (closed cases)")

print("\nTrain Slow Rate by Borough:")
train_borough = train_eligible.groupby("borough")["slow_resolution_label"].agg(["count", "mean"])
train_borough["mean"] = train_borough["mean"].map(lambda x: f"{x:.1%}")
train_borough.columns = ["eligible_rows", "slow_rate"]
print(train_borough.to_string())

# 3. Audit Validation Partition
val_eligible = df_combined[(df_combined["split"] == "val") & (df_combined["eligible_for_modeling"])]
val_total = len(val_eligible)
val_slow = (val_eligible["slow_resolution_label"] == 1).sum()
val_not_slow = (val_eligible["slow_resolution_label"] == 0).sum()
val_slow_rate = val_slow / val_total
val_median_hours = val_eligible["resolution_hours"].dropna().median()

print("\n=== Validation Partition Health (Jan–Feb 2024; Peak Winter) ===")
print(f"Eligible records       : {val_total:,} (0 clock anomalies)")
print(f"Not Slow (Y = 0)       : {val_not_slow:,} ({1 - val_slow_rate:.1%})")
print(f"Slow     (Y = 1)       : {val_slow:,} ({val_slow_rate:.1%})")
print(f"Median resolution time : {val_median_hours:.1f} hours (closed cases)")

print("\nValidation Slow Rate by Borough:")
val_borough = val_eligible.groupby("borough")["slow_resolution_label"].agg(["count", "mean"])
val_borough["mean"] = val_borough["mean"].map(lambda x: f"{x:.1%}")
val_borough.columns = ["eligible_rows", "slow_rate"]
print(val_borough.to_string())

# 4. Test Set Integrity (Row counts verified, target sealed)
test_records = df_combined[df_combined["split"] == "test"]
test_eligible_count = test_records["eligible_for_modeling"].sum()

print("\n=== Test Partition Status (Mar–May 2024; Sealed) ===")
print(f"Total test records     : {len(test_records):,}")
print(f"Eligible records       : {test_eligible_count:,}")
print("[HELD OUT] Test set labels remain sealed and uninspected until final model evaluation.")

=== Train Partition Health (Oct–Dec 2023) ===
Eligible records       : 106,026 (162 clock anomalies excluded)
Not Slow (Y = 0)       : 78,195 (73.8%)
Slow     (Y = 1)       : 27,831 (26.2%)
Median resolution time : 41.8 hours (closed cases)

Train Slow Rate by Borough:
               eligible_rows slow_rate
borough                               
BRONX                  36540     22.4%
BROOKLYN               28157     20.5%
MANHATTAN              25181     36.4%
QUEENS                 14797     28.8%
STATEN ISLAND           1351     34.3%

=== Validation Partition Health (Jan–Feb 2024; Peak Winter) ===
Eligible records       : 81,641 (0 clock anomalies)
Not Slow (Y = 0)       : 69,456 (85.1%)
Slow     (Y = 1)       : 12,185 (14.9%)
Median resolution time : 32.2 hours (closed cases)

Validation Slow Rate by Borough:
               eligible_rows slow_rate
borough                               
BRONX                  30139      8.3%
BROOKLYN               20915     16.1%
MANHATTAN          

## Section 7: Serialization & Provenance Sidecars

We serialize the feature matrix and quarantined label tables into `data/labelled/`, generating cryptographic SHA-256 sidecars that document source provenance, schema boundaries, and partition distributions.

In [9]:
from datetime import datetime, timezone

LABELLED_DIR = REPO_ROOT / "data" / "labelled"
LABELLED_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = LABELLED_DIR / "heat_2023_2024_features_at_intake.json"
FEATURES_META_PATH = LABELLED_DIR / "heat_2023_2024_features_at_intake.meta.json"

LABELS_PATH = LABELLED_DIR / "heat_2023_2024_slow_resolution_labels.json"
LABELS_META_PATH = LABELLED_DIR / "heat_2023_2024_slow_resolution_labels.meta.json"

now_iso = datetime.now(timezone.utc).isoformat()

# 1. Serialize Features Artifact
print(f"Writing features artifact to {FEATURES_PATH}...")
features_bytes = df_features.to_json(orient="records", indent=2).encode("utf-8")
FEATURES_PATH.write_bytes(features_bytes)
features_sha256 = hashlib.sha256(features_bytes).hexdigest()
features_mb = len(features_bytes) / (1024 * 1024)
print(f"  Wrote {features_mb:.1f} MB (SHA-256: {features_sha256[:16]}...)")

features_sidecar = {
    "built_at": now_iso,
    "source_raw_file": str(RAW_DATA_PATH.relative_to(REPO_ROOT)),
    "raw_source_sha256": computed_sha256,
    "rows": len(df_features),
    "sha256": features_sha256,
    "size_bytes": len(features_bytes),
    "split_counts": {k: int(v) for k, v in df_features["split"].value_counts().items()},
    "audit_columns": AUDIT_COLUMNS,
    "model_input_columns": MODEL_INPUT_COLUMNS,
    "forbidden_fields_enforced": FORBIDDEN_OUTCOME_FIELDS + FORBIDDEN_PII_GEOGRAPHY,
    "closure_fields_written": False,
    "notes": "Strictly intake-only features. Split column travels for chronological routing; never a model input."
}
FEATURES_META_PATH.write_text(json.dumps(features_sidecar, indent=2), encoding="utf-8")
print(f"  Wrote features sidecar to {FEATURES_META_PATH}")

# 2. Serialize Labels Artifact
print(f"\nWriting labels artifact to {LABELS_PATH}...")
labels_bytes = df_labels.to_json(orient="records", indent=2).encode("utf-8")
LABELS_PATH.write_bytes(labels_bytes)
labels_sha256 = hashlib.sha256(labels_bytes).hexdigest()
labels_mb = len(labels_bytes) / (1024 * 1024)
print(f"  Wrote {labels_mb:.1f} MB (SHA-256: {labels_sha256[:16]}...)")

labels_sidecar = {
    "built_at": now_iso,
    "source_raw_file": str(RAW_DATA_PATH.relative_to(REPO_ROOT)),
    "raw_source_sha256": computed_sha256,
    "rows": len(df_labels),
    "sha256": labels_sha256,
    "size_bytes": len(labels_bytes),
    "threshold_hours": float(FROZEN_TAU_HOURS),
    "threshold_status": "project-defined baseline (quiet-window p90 benchmark) - not an SLA",
    "eligible_for_modeling_count": int(df_labels["eligible_for_modeling"].sum()),
    "quarantined_clock_anomalies_count": int(df_labels["is_clock_anomaly"].sum()),
    "mature_unclosed_tickets_count": int(unclosed_mask.sum()),
    "train_metrics": {
        "eligible": int(train_total),
        "slow_count": int(train_slow),
        "slow_rate": round(float(train_slow_rate), 4),
        "median_hours": round(float(train_median_hours), 2)
    },
    "val_metrics": {
        "eligible": int(val_total),
        "slow_count": int(val_slow),
        "slow_rate": round(float(val_slow_rate), 4),
        "median_hours": round(float(val_median_hours), 2)
    },
    "test_metrics_status": "held out / sealed until final evaluation",
    "columns": list(df_labels.columns),
}
LABELS_META_PATH.write_text(json.dumps(labels_sidecar, indent=2), encoding="utf-8")
print(f"  Wrote labels sidecar to {LABELS_META_PATH}")

# 3. Final Verification
assert FEATURES_PATH.exists() and FEATURES_META_PATH.exists()
assert LABELS_PATH.exists() and LABELS_META_PATH.exists()
print("\n[PASS] All 2023-2024 heating season labeled artifacts written and verified successfully.")

Writing features artifact to C:\Users\mejdi\Documents\resolveiq\data\labelled\heat_2023_2024_features_at_intake.json...
  Wrote 94.7 MB (SHA-256: d7895fa4663edae3...)
  Wrote features sidecar to C:\Users\mejdi\Documents\resolveiq\data\labelled\heat_2023_2024_features_at_intake.meta.json

Writing labels artifact to C:\Users\mejdi\Documents\resolveiq\data\labelled\heat_2023_2024_slow_resolution_labels.json...
  Wrote 49.4 MB (SHA-256: ccb686ec9fdb33c0...)
  Wrote labels sidecar to C:\Users\mejdi\Documents\resolveiq\data\labelled\heat_2023_2024_slow_resolution_labels.meta.json

[PASS] All 2023-2024 heating season labeled artifacts written and verified successfully.
